In [11]:
from snowflake.snowpark import Session
from snowflake.ml.jobs import submit_directory
from poc.utils import create_session, get_connection_config, get_training_config, get_inference_config
import subprocess
import os
import re

conn_cfg = get_connection_config()
train_cfg = get_training_config()
infer_cfg = get_inference_config()

session = create_session()

train = "container" if train_cfg["mljob"] else "warehouse"
infer = "container" if infer_cfg["mljob"] else "warehouse"

def parse_run_id(stdout: str) -> str:
    match = re.search(r"RUN_ID=(.+)", stdout)
    return match.group(1).strip() if match else ""

In [2]:
from datetime import datetime, timezone
train_start = datetime.now(timezone.utc)
if train == "warehouse":
    result = subprocess.run(["python", os.getcwd()+"/poc/train_warehouse.py"], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    train_run_id = parse_run_id(result.stdout)
elif train == "container":
    train_job = submit_directory(
        dir_path="poc/",
        compute_pool=train_cfg["compute_pool_name"],
        entrypoint="train.py",
        stage_name=conn_cfg["stage_artifacts"],
        target_instances=train_cfg["target_cluster_size"],
        session=session,
    )
    train_job.wait()
    logs = train_job.get_logs()
    print(logs)
    train_run_id = parse_run_id(logs)
train_end = datetime.now(timezone.utc)
print(f"train_run_id: {train_run_id}")

Compute pool busy (4/5 nodes in use, 5 nodes required). Job execution may be delayed.


Connected: "UR20380"
🔄 Preparing data for 100 partitions
   Test percentage: 10%
   Train rows: 65,700
   Test rows: 7,300
----------------------------------------------------------------------------------------------------------------------------------
|"name"                                              |"size"  |"md5"                             |"last_modified"                |
----------------------------------------------------------------------------------------------------------------------------------
|ml_stage/train_features/P0000/data_01c350ae-000...  |30753   |76e8e84549cb2ed33ce826ae74487975  |Fri, 27 Mar 2026 20:30:06 GMT  |
|ml_stage/train_features/P0001/data_01c350ae-000...  |31425   |9c0bfaf6e5532a90e86c2608f12b417d  |Fri, 27 Mar 2026 20:30:06 GMT  |
|ml_stage/train_features/P0002/data_01c350ae-000...  |30790   |30b8d91bd28f11c8727c5cf4e28fddf1  |Fri, 27 Mar 2026 20:30:06 GMT  |
|ml_stage/train_features/P0003/data_01c350ae-000...  |31377   |0af9dae31be9c106c2143270ff0a

In [3]:
infer_start = datetime.now(timezone.utc)
if infer == "warehouse":
    result = subprocess.run(["python", os.getcwd()+"/poc/infer_warehouse.py"], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    infer_run_id = parse_run_id(result.stdout)
elif infer == "container":
    infer_job = submit_directory(
        dir_path="poc/",
        compute_pool=infer_cfg["compute_pool_name"],
        entrypoint="infer.py",
        stage_name=conn_cfg["stage_artifacts"],
        target_instances=infer_cfg["target_cluster_size"],
        session=session,
    )
    infer_job.wait()
    logs = infer_job.get_logs()
    print(logs)
    infer_run_id = parse_run_id(logs)
infer_end = datetime.now(timezone.utc)
print(f"infer_run_id: {infer_run_id}")

Connected: "SFSENORTHAMERICA-AFERAS_AWS1"
📋 Loading model from registry
   Model: MMT_POC
   Version: SOFT_SHEEP_1
   Rows to score: 7300

🚀 Running inference via Model Registry
   Predictions written to PREDICTIONS table

📈 Sample Predictions:
-----------------------------------------------------------------------
|"PARTITION_ID"  |"TS"                 |"PREDICTION"       |"TARGET"  |
-----------------------------------------------------------------------
|P0017           |2026-01-22 00:00:00  |83.79773712158203  |84.34     |
|P0017           |2026-02-05 00:00:00  |82.68391418457031  |92.01     |
|P0017           |2026-02-04 00:00:00  |89.62174224853516  |88.63     |
|P0017           |2026-02-03 00:00:00  |88.4107437133789   |85.34     |
|P0017           |2026-02-02 00:00:00  |86.70339965820312  |89.1      |
-----------------------------------------------------------------------

RUN_ID=inference_20260327_2036

infer_run_id: inference_20260327_2036


## Job Metrics: Time and Credit Consumption ESTIMATION

In [4]:
INSTANCE_FAMILY_CREDITS_PER_HOUR = {
    "CPU_X64_XS": 0.06,
    "CPU_X64_S": 0.19,
    "CPU_X64_M": 0.38,
    "CPU_X64_SL": 0.86,
    "CPU_X64_L": 1.72,
    "HIGHMEM_X64_S": 0.38,
    "HIGHMEM_X64_M": 1.72,
    "HIGHMEM_X64_SL": 5.66,
    "HIGHMEM_X64_L": 7.62,
    "GPU_NV_XS": 1.0,
    "GPU_NV_S": 1.5,
    "GPU_NV_SM": 3.0,
    "GPU_NV_M": 6.0,
    "GPU_NV_2M": 9.0,
    "GPU_NV_3M": 16.0,
    "GPU_NV_L": 18.5,
    "GPU_NV_SL": 32.0,
}

WAREHOUSE_SIZE_CREDITS_PER_HOUR = {
    "X-Small": 1, "Small": 2, "Medium": 4, "Large": 8,
    "X-Large": 16, "2X-Large": 32, "3X-Large": 64, "4X-Large": 128,
    "5X-Large": 256, "6X-Large": 512,
}

In [12]:
train_duration = (train_end - train_start).total_seconds()
infer_duration = (infer_end - infer_start).total_seconds()

def calc_container_credits(duration_secs, instance_family, num_nodes):
    rate = INSTANCE_FAMILY_CREDITS_PER_HOUR.get(instance_family, 0)
    return (duration_secs / 3600) * rate * num_nodes

train_container_credits = calc_container_credits(train_duration, train_cfg["instance_family"], train_cfg["target_cluster_size"]) if train=="container" else 0
infer_container_credits = calc_container_credits(infer_duration, infer_cfg["instance_family"], infer_cfg["target_cluster_size"]) if infer=="container" else 0

def get_wh_credits(session, query_tag):
    df = session.sql(f"""
        SELECT WAREHOUSE_SIZE, SUM(EXECUTION_TIME) / 1000 AS TOTAL_EXEC_SECS
        FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
        WHERE QUERY_TAG = '{query_tag}'
          AND WAREHOUSE_SIZE IS NOT NULL
        GROUP BY WAREHOUSE_SIZE
    """).to_pandas()
    credits = 0
    for _, row in df.iterrows():
        rate = WAREHOUSE_SIZE_CREDITS_PER_HOUR.get(row['WAREHOUSE_SIZE'], 0)
        credits += (row['TOTAL_EXEC_SECS'] / 3600) * rate
    return credits, df

train_wh_credits, wh_train_df = get_wh_credits(session, train_run_id)
infer_wh_credits, wh_infer_df = get_wh_credits(session, infer_run_id)
print(f"train_wh_credits: {train_wh_credits}, infer_wh_credits: {infer_wh_credits}")

train_wh_credits: 0.052596666666666674, infer_wh_credits: 0.008973333333333333


In [13]:
import pandas as pd

summary_df = pd.DataFrame({
    'Job': ['Training', 'Inference', 'Total'],
    'Instance Family': [train_cfg.get("instance_family",""), infer_cfg.get("instance_family",""), '-'],
    'Nodes': [train_cfg.get("target_cluster_size",""), infer_cfg.get("target_cluster_size",""), '-'],
    'Duration (seconds)': [round(train_duration, 2), round(infer_duration, 2), round(train_duration + infer_duration, 2)],
    'Duration (minutes)': [round(train_duration / 60, 2), round(infer_duration / 60, 2), round((train_duration + infer_duration) / 60, 2)],
    'Container Credits (est)': [round(train_container_credits, 6), round(infer_container_credits, 6), round(train_container_credits + infer_container_credits, 6)],
    'Warehouse Credits (est)': [round(train_wh_credits, 6), round(infer_wh_credits, 6), round(train_wh_credits + infer_wh_credits, 6)],
    'Total Credits (est)': [round(train_container_credits + train_wh_credits, 6), round(infer_container_credits + infer_wh_credits, 6), round(train_container_credits + infer_container_credits + train_wh_credits + infer_wh_credits, 6)]
})
summary_df

,Job,Instance Family,Nodes,Duration (seconds),Duration (minutes),Container Credits (est),Warehouse Credits (est),Total Credits (est)
0,Training,CPU_X64_S,5,496.48,8.27,0.131015,0.052597,0.183611
1,Inference,,,24.08,0.40,0.000000,0.008973,0.008973
2,Total,-,-,520.56,8.68,0.131015,0.061570,0.192585
